# API Presensi: Face Landmark Challenge → ArcFace

Versi ini menghapus CNN anti-spoof CASIA-FASD. Liveness diperiksa dengan **challenge-response kanan/kiri berbasis MediaPipe Face Landmarker**, lalu frame terbaik diteruskan ke ArcFace + SVM/centroid.

Untuk prototype backend, endpoint menerima klip singkat dan challenge yang diminta server. Untuk mobile production, logic landmark dapat dijalankan di Flutter secara lokal; backend tetap dapat melakukan re-check pada frame/klip singkat bila ingin menambah lapisan keamanan.


In [ ]:
!pip -q install fastapi uvicorn pyngrok python-multipart opencv-python-headless mediapipe deepface scikit-learn


In [ ]:
import io, json, os, pickle, tempfile, getpass, secrets, time, asyncio, urllib.request
from pathlib import Path
import cv2
import numpy as np
import mediapipe as mp
from fastapi import FastAPI, File, UploadFile, HTTPException, Form
from fastapi.responses import JSONResponse
from deepface import DeepFace

ARTIFACT_DIR=Path('/content/model')
FACE_CLASSIFIER_PATH=ARTIFACT_DIR/'face_classifier.pkl'
LABEL_ENCODER_PATH=ARTIFACT_DIR/'label_encoder.pkl'
CENTROIDS_PATH=ARTIFACT_DIR/'face_centroids.npy'
CENTROID_LABELS_PATH=ARTIFACT_DIR/'centroid_labels.npy'
FACE_CONFIG_PATH=ARTIFACT_DIR/'face_recognition_config.json'
required=[FACE_CLASSIFIER_PATH,LABEL_ENCODER_PATH,CENTROIDS_PATH,CENTROID_LABELS_PATH]
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing ArcFace artifact(s): '+', '.join(missing))
with open(FACE_CLASSIFIER_PATH,'rb') as f: face_classifier=pickle.load(f)
with open(LABEL_ENCODER_PATH,'rb') as f: label_encoder=pickle.load(f)
centroids=np.load(CENTROIDS_PATH,allow_pickle=True).astype(np.float32)
centroid_labels=np.load(CENTROID_LABELS_PATH,allow_pickle=True)
face_cfg=json.loads(FACE_CONFIG_PATH.read_text(encoding='utf-8')) if FACE_CONFIG_PATH.exists() else {}
OPEN_SET_THRESHOLD=float(face_cfg.get('open_set_threshold',0.5))
MODEL_NAME=face_cfg.get('model_name','ArcFace')
DETECTOR_BACKEND=face_cfg.get('detector_backend','ssd')

LANDMARK_MODEL=Path('/content/face_landmarker.task')
LANDMARK_URL='https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task'
if not LANDMARK_MODEL.exists(): urllib.request.urlretrieve(LANDMARK_URL,LANDMARK_MODEL)

mp_options=mp.tasks.vision.FaceLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(LANDMARK_MODEL)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
landmarker=mp.tasks.vision.FaceLandmarker.create_from_options(mp_options)

LEFT_EYE_OUTER=33; RIGHT_EYE_OUTER=263; NOSE_TIP=1
CALIBRATION_FRAMES=8
HOLD_FRAMES=3
LEFT_DELTA=-0.10
RIGHT_DELTA=0.10
MAX_VIDEO_SECONDS=8
MAX_UPLOAD_BYTES=25*1024*1024
SAMPLED_FRAMES=24
ALLOWED_SUFFIXES={'.mp4','.avi','.mov','.mkv','.mpeg','.mpg'}


In [ ]:
app=FastAPI(title='Presensi AI: Face Landmark Liveness + ArcFace',version='2.0.0')

def _xy(face_landmarks,idx):
    p=face_landmarks[idx]
    return np.array([p.x,p.y],dtype=np.float32)

def yaw_proxy(face_landmarks):
    left_eye=_xy(face_landmarks,LEFT_EYE_OUTER); right_eye=_xy(face_landmarks,RIGHT_EYE_OUTER); nose=_xy(face_landmarks,NOSE_TIP)
    mid=(left_eye+right_eye)/2.0; dist=float(np.linalg.norm(right_eye-left_eye))
    return None if dist<1e-6 else float((nose[0]-mid[0])/dist)

def get_video_frames(video_path):
    cap=cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): raise ValueError('video_decode_failed')
    fps=cap.get(cv2.CAP_PROP_FPS) or 0; total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration=(total/fps) if fps>0 else 0
    if total<2 or duration>MAX_VIDEO_SECONDS:
        cap.release(); raise ValueError('video_too_short_or_too_long')
    indices=np.linspace(0,total-1,num=min(SAMPLED_FRAMES,total),dtype=int)
    frames=[]
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(idx)); ok,frame=cap.read()
        if ok and frame is not None: frames.append(frame)
    cap.release()
    if len(frames)<CALIBRATION_FRAMES: raise ValueError('insufficient_frames')
    return frames,fps

def landmark_yaw(frame,timestamp_ms):
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    img=mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb)
    r=landmarker.detect_for_video(img,timestamp_ms)
    if not r.face_landmarks: return None
    return yaw_proxy(r.face_landmarks[0])

def check_challenge(frames,fps,target):
    values=[]
    for i,frame in enumerate(frames): values.append(landmark_yaw(frame,int(i*1000/max(fps,1))))
    valid=[v for v in values if v is not None]
    if len(valid)<CALIBRATION_FRAMES: return {'passed':False,'reason':'face_not_stable_or_not_detected'}
    baseline=float(np.median(valid[:CALIBRATION_FRAMES]))
    deltas=[None if v is None else v-baseline for v in values[CALIBRATION_FRAMES:]]
    for i in range(max(0,len(deltas)-HOLD_FRAMES+1)):
        w=deltas[i:i+HOLD_FRAMES]
        if all(d is not None for d in w):
            ok=all(d<=LEFT_DELTA for d in w) if target=='LEFT' else all(d>=RIGHT_DELTA for d in w)
            if ok:
                return {'passed':True,'target':target,'baseline':baseline,'peak_delta':float(min(w) if target=='LEFT' else max(w)),'face_valid_frames':len(valid)}
    return {'passed':False,'target':target,'baseline':baseline,'peak_delta':None,'face_valid_frames':len(valid)}

def l2_normalize(v):
    v=np.asarray(v,dtype=np.float32).reshape(-1); n=float(np.linalg.norm(v))
    if not np.isfinite(n) or n<=1e-12: raise ValueError('invalid_embedding')
    return v/n

def extract_arcface_embedding(frame_bgr):
    rgb=cv2.cvtColor(frame_bgr,cv2.COLOR_BGR2RGB)
    reps=DeepFace.represent(img_path=rgb,model_name=MODEL_NAME,detector_backend=DETECTOR_BACKEND,
                            enforce_detection=True,align=True,normalization='base',silent=True)
    if not reps: raise ValueError('face_not_detected_for_recognition')
    return l2_normalize(reps[0]['embedding'])

def recognize_embedding(embedding):
    similarities=centroids@embedding
    i=int(np.argmax(similarities)); sim=float(similarities[i]); label=centroid_labels[i]
    recognized=sim>=OPEN_SET_THRESHOLD
    return {'recognized':recognized,'identity':label.item() if hasattr(label,'item') else str(label),'similarity':sim}


In [ ]:
@app.post('/attendance')
async def attendance(
    file: UploadFile=File(...),
    challenge: str=Form(...),
    challenge_id: str=Form(...),
):
    target=challenge.upper().strip()
    if target not in {'LEFT','RIGHT'}:
        raise HTTPException(status_code=400,detail={'code':'invalid_challenge','allowed':['LEFT','RIGHT']})
    suffix=Path(file.filename or 'recording.mp4').suffix.lower()
    if suffix not in ALLOWED_SUFFIXES: raise HTTPException(status_code=415,detail={'code':'unsupported_video'})
    contents=await file.read(MAX_UPLOAD_BYTES+1)
    if len(contents)>MAX_UPLOAD_BYTES: raise HTTPException(status_code=413,detail={'code':'video_too_large'})
    temp_path=None
    try:
        with tempfile.NamedTemporaryFile(suffix=suffix,delete=False) as tmp:
            tmp.write(contents); temp_path=tmp.name
        frames,fps=get_video_frames(temp_path)
        live=check_challenge(frames,fps,target)
        if not live['passed']:
            return JSONResponse(status_code=422,content={'status':'rejected','valid':False,'reason':'liveness_failed','challenge_id':challenge_id,**live})
        # Pick a middle/last valid frame for recognition. For production, choose the sharpest face frame.
        best_frame=frames[-1]
        embedding=extract_arcface_embedding(best_frame)
        result=recognize_embedding(embedding)
        if not result['recognized']:
            return JSONResponse(status_code=404,content={'status':'rejected','valid':False,'reason':'unknown_identity','challenge_id':challenge_id,'liveness':live,'similarity':round(result['similarity'],4),'open_set_threshold':OPEN_SET_THRESHOLD})
        return {'status':'success','valid':True,'challenge_id':challenge_id,'liveness':live,'identity':result['identity'],'similarity':round(result['similarity'],4)}
    except ValueError as e:
        raise HTTPException(status_code=422,detail={'code':str(e)})
    finally:
        if temp_path:
            try: os.unlink(temp_path)
            except OSError: pass


## Jalankan tunnel publik sementara

Untuk Colab saja. Sebelum digunakan di aplikasi nyata, tambahkan autentikasi, rate limit, HTTPS, validasi nonce/challenge expiry, dan pencatatan audit.


In [ ]:
import uvicorn
from pyngrok import ngrok
ngrok_token=getpass.getpass('Masukkan ngrok authtoken (tidak disimpan di notebook): ')
ngrok.set_auth_token(ngrok_token)
public_tunnel=ngrok.connect(8001,'http')
print('Public API:',public_tunnel.public_url)
print('Swagger UI:',public_tunnel.public_url+'/docs')
config=uvicorn.Config(app=app,host='0.0.0.0',port=8001,log_level='info')
server=uvicorn.Server(config)
await server.serve()


## Catatan integrasi Flutter

Server sebaiknya mengeluarkan `challenge_id` + challenge acak, misalnya `RIGHT`. Flutter menjalankan MediaPipe Face Landmarker secara live dan menampilkan instruksi kepada pegawai. Untuk prototype, klip hasil challenge dapat dikirim ke endpoint di atas agar server melakukan re-check.

Untuk versi mobile yang lebih ringan, Face Landmarker dapat dijalankan lokal pada live stream dan hanya frame terbaik yang dikirim ke backend untuk ArcFace. Namun, jangan menganggap field `liveness_passed=true` dari client sebagai bukti keamanan yang kuat; client yang dimodifikasi dapat memalsukan hasil. Server-issued nonce dan re-check adalah lapisan yang lebih aman.
